In [ ]:
#import 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np


In [ ]:
#load processed data 
df_particles=pd.read_csv('processed_data/cleaned_fine_particles_data.csv')
df_no2=pd.read_csv('processed_data/cleaned_nitrogen_dioxide_data.csv')
df_traffic=pd.read_csv('processed_data/cleaned_traffic_data.csv')

In [ ]:
print("RAW TRAFFIC TYPE:", type(df_traffic['DateTime'].iloc[0]))
print("RAW PARTICLES TYPE:", type(df_particles['TimePeriod'].iloc[0]))

In [ ]:
#add text col of boro

# 1hot encoding cols 
cols = ['Borough_Manhattan', 'Borough_Brooklyn', 'Borough_Queens', 'Borough_Staten Island']

# add the text for the 4 defined (particles)
df_particles['Borough'] = df_particles[cols].idxmax(axis=1).str.replace('Borough_', '')

# add manhattan (particles)
df_particles.loc[df_particles[cols].sum(axis=1) == 0, 'Borough'] = 'Bronx'

#save
df_particles.to_csv('processed_data/cleaned_fine_particles_data.csv', index=False)

# add the text for the 4 defined (no2)
df_no2['Borough'] = df_no2[cols].idxmax(axis=1).str.replace('Borough_', '')

# add manhattan (no2)
df_no2.loc[df_no2[cols].sum(axis=1) == 0, 'Borough'] = 'Bronx'

#save 
df_no2.to_csv('processed_data/cleaned_nitrogen_dioxide_data.csv', index=False)

traffic_cols=['Boro_Manhattan', 'Boro_Brooklyn', 'Boro_Queens', 'Boro_Staten Island']

# add the text for the 4 defined (traffic)
df_traffic['Boro'] = df_traffic[traffic_cols].idxmax(axis=1).str.replace('Boro_', '')

# add manhattan (traffic)
df_traffic.loc[df_traffic[traffic_cols].sum(axis=1) == 0, 'Boro'] = 'Bronx'

#save 
df_traffic.to_csv('processed_data/cleaned_traffic_data.csv', index=False)

In [ ]:
#sql stuff
import sqlite3
conn = sqlite3.connect('main.db')

#update the staten island col name + date
df_particles = df_particles.rename(columns={
    'Borough_Staten Island': 'Borough_Staten_Island', 
    'TimePeriod': 'Date'
})
df_no2 = df_no2.rename(columns={
    'Borough_Staten Island': 'Borough_Staten_Island',
    'TimePeriod': 'Date'
})
#update the names for traffic 
df_traffic = df_traffic.rename(columns={
    'Boro_Manhattan': 'Borough_Manhattan',
    'Boro_Brooklyn': 'Borough_Brooklyn',
    'Boro_Queens': 'Borough_Queens',
    'Boro_Staten Island': 'Borough_Staten_Island',
    'DateTime': 'Date',
    'Boro': 'Borough'
})

#cleaning the datetime cols 

#get the fulldatetime object 
df_traffic['Date'] = pd.to_datetime(df_traffic['Date'])
df_particles['Date'] = pd.to_datetime(df_particles['Date'])
df_no2['Date'] = pd.to_datetime(df_no2['Date'])

#add a season to traffic using the month 
df_traffic['Month'] = df_traffic['Date'].dt.month
rules = [
    df_traffic['Month'].isin([12, 1, 2]), # winter
    df_traffic['Month'].isin([6, 7, 8])   # summer
]
choices = ['Winter', 'Summer']
df_traffic['Season'] = np.select(rules, choices, default='Other')

# get the year 
df_traffic['Date'] = df_traffic['Date'].dt.year
df_particles['Date'] = df_particles['Date'].dt.year
df_no2['Date'] = df_no2['Date'].dt.year

#fixing structure

#particles 
# base_cols = ['Date', 'Borough', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island']
p_cols=['Date', 'Borough', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island', 
          'Annual mean mcg/m3', 'Annual mean mcg/m3_log']

df_p_val = df_particles.melt(id_vars=p_cols, value_vars=['Summer mean mcg/m3', 'Winter mean mcg/m3'], 
                             var_name='Season', value_name='Fine_Particles_Value')
df_p_val['Season'] = df_p_val['Season'].str.replace(' mean mcg/m3', '').str.strip()

df_p_log = df_particles.melt(id_vars=['Date', 'Borough'], value_vars=['Summer mean mcg/m3_log', 'Winter mean mcg/m3_log'], 
                             var_name='Season', value_name='Fine_Particles_Log')
df_p_log['Season'] = df_p_log['Season'].str.replace(' mean mcg/m3_log', '').str.strip()

df_particles_final = df_p_val.merge(df_p_log, on=['Date', 'Borough', 'Season'])

#no2
n_cols=['Date', 'Borough', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island', 
          'Annual mean ppb', 'Annual mean ppb_log']

df_n_val = df_no2.melt(id_vars=n_cols, value_vars=['Summer mean ppb', 'Winter mean ppb'], 
                       var_name='Season', value_name='NO2_Value')
df_n_val['Season'] = df_n_val['Season'].str.replace(' mean ppb', '').str.strip()

df_n_log = df_no2.melt(id_vars=['Date', 'Borough'], value_vars=['Summer mean ppb_log', 'Winter mean ppb_log'], 
                       var_name='Season', value_name='NO2_Log')
df_n_log['Season'] = df_n_log['Season'].str.replace(' mean ppb_log', '').str.strip()

df_no2_final = df_n_val.merge(df_n_log, on=['Date', 'Borough', 'Season'])

# df_no2_new = df_no2.melt(
#     id_vars=base_cols,
#     value_vars=['Summer mean ppb', 'Winter mean ppb'],
#     var_name='Season',
#     value_name='NO2_Value'
# )

# df_particles_new['Season'] = df_particles_new['Season'].str.replace(' mean mcg/m3', '')
# df_no2_new['Season'] = df_no2_new['Season'].str.replace(' mean ppb', '')

#create tables

df_traffic.to_sql('traffic_volume', conn, if_exists='replace', index=False)
df_particles_final.to_sql('fine_particles', conn, if_exists='replace', index=False)
df_no2_final.to_sql('nitrogen_dioxide', conn, if_exists='replace', index=False)


#test 
print("testing...")
print("Traffic Columns:", df_traffic.columns.tolist())
print("Particles Columns:", df_particles.columns.tolist())
print("NO2 Columns:", df_no2.columns.tolist())
print("...testing")

#end of test

#master table 
master_query = """
CREATE TABLE master_data AS
SELECT 
    t.Date AS Year, 
    t.Month, 
    t.Borough, 
    t.Season, 
    t.Vol AS Traffic_Volume, 
    t.Vol_log AS Traffic_Volume_Log,
    p.Fine_Particles_Value, 
    p.Fine_Particles_Log,
    p."Annual mean mcg/m3" AS Annual_Particles_Mean,
    p."Annual mean mcg/m3_log" AS Annual_Particles_Log,
    n.NO2_Value,
    n.NO2_Log,
    n."Annual mean ppb" AS Annual_NO2_Mean,
    n."Annual mean ppb_log" AS Annual_NO2_Log,
    t.Borough_Manhattan,
    t.Borough_Brooklyn,
    t.Borough_Queens,
    t.Borough_Staten_Island
FROM traffic_volume t
JOIN fine_particles p 
    ON t.Date = p.Date AND t.Borough = p.Borough AND t.Season = p.Season
JOIN nitrogen_dioxide n
    ON t.Date = n.Date AND t.Borough = n.Borough AND t.Season = n.Season;
"""

conn.execute("DROP TABLE IF EXISTS master_data") 
conn.execute(master_query)

In [ ]:
# query for linear regression
query = "SELECT * FROM master_data"
df_main=pd.read_sql(query, conn)
#print(df_main.head())
#print(df_main.info())
conn.close()

In [ ]:

df_main=df_main.sort_values(by=['Borough', 'Year', 'Season'])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
#no2 model 1
features_no2 = [
    'Traffic_Volume', 'Borough_Manhattan', 'Borough_Brooklyn', 'Borough_Queens', 'Borough_Staten_Island',
]
X_no2 = df_main[features_no2]
y_no2 = df_main['NO2_Log']

X_train_no2, X_test_no2, y_train_no2, y_test_no2 = train_test_split(X_no2, y_no2, test_size=0.2, random_state=42)

model_no2 = LinearRegression()
model_no2.fit(X_train_no2, y_train_no2)
y_pred_no2 = model_no2.predict(X_test_no2)

In [ ]:
# fine particles model 2
features_p = [
    'Traffic_Volume', 'Borough_Manhattan', 'Borough_Brooklyn', 'Borough_Queens', 'Borough_Staten_Island',
]

X_particles = df_main[features_p]
y_particles = df_main['Fine_Particles_Log']

X_train_particles, X_test_particles, y_train_particles, y_test_particles = train_test_split(X_particles, y_particles, test_size=0.2, random_state=42)

model_particles = LinearRegression()
model_particles.fit(X_train_particles, y_train_particles)
y_pred_particles = model_particles.predict(X_test_particles)

In [ ]:
#show linear regression no2 
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(y_test_no2, y_pred_no2, alpha=0.5, color='blue')

plt.plot([y_test_no2.min(), y_test_no2.max()], [y_test_no2.min(), y_test_no2.max()], 'r--', lw=2)

plt.xlabel('Actual NO2 (Log Values)')
plt.ylabel('Predicted NO2 (Log Values)')
plt.title('Linear Regression: Actual vs. Predicted NO2')
plt.show()

In [ ]:

#show linear regression fine particles 
plt.figure(figsize=(10, 6))
plt.scatter(y_test_particles, y_pred_particles, alpha=0.5, color='blue')

plt.plot([y_test_particles.min(), y_test_particles.max()], [y_test_particles.min(), y_test_particles.max()], 'r--', lw=2)

plt.xlabel('Actual Fine Particles (Log Values)')
plt.ylabel('Predicted Fine Particles (Log Values)')
plt.title('Linear Regression: Actual vs. Predicted Fine Particles')
plt.show()

In [ ]:
#metrics 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
#no2 metrics 
r2_no2 = r2_score(y_test_no2, y_pred_no2)
rmse_no2 = np.sqrt(mean_squared_error(y_test_no2, y_pred_no2))
mae_no2 = mean_absolute_error(y_test_no2, y_pred_no2)
mape_no2 = np.mean(np.abs((y_test_no2 - y_pred_no2) / y_test_no2)) * 100
print("NO2 Linear Regression Model Metrics:")
print(f"R-squared: {r2_no2:.4f}")
print(f"RMSE: {rmse_no2:.4f}")
print(f"MAE (high precision due to log use): {mae_no2:.6f}")
print(f"MAPE(high precision due to log use): {mape_no2:.6f}%")
print(" ")

#fine particles metrics 
r2_particles = r2_score(y_test_particles, y_pred_particles)
rmse_particles = np.sqrt(mean_squared_error(y_test_particles, y_pred_particles))
mae_particles = mean_absolute_error(y_test_particles, y_pred_particles)
mape_particles = np.mean(np.abs((y_test_particles - y_pred_particles) / y_test_particles)) * 100
print("Fine Particles Linear Regression Model Metrics:")
print(f"R-squared: {r2_particles:.4f}")
print(f"RMSE: {rmse_particles:.4f}")
print(f"MAE(high precision due to log use): {mae_particles:.6f}")
print(f"MAPE (high precision due to log use): {mape_particles:.6f}%")

Time Series Analysis

In [ ]:
#df for time-series analysis of particles and traffic volume 
df_pt = df_main[['Year','Month', 'Borough', 'Fine_Particles_Log', 'Traffic_Volume_Log']].copy()
df_pt['Date'] = pd.to_datetime(df_pt['Year'].astype(str) + '-' + \
                                  df_pt['Month'].astype(str))
df_pt = df_pt.set_index('Date')
df_pt = df_pt.drop(columns=['Year', 'Month'])
df_pt = df_pt.sort_index()
df_pt.head()


In [ ]:
#df for time-series analysis of no2 and traffic volume
df_n2t = df_main[['Month', 'Year', 'Borough', 'NO2_Log', 'Traffic_Volume_Log']].copy()
df_n2t['Date'] = pd.to_datetime(df_n2t['Year'].astype(str) + '-' + \
                                  df_n2t['Month'].astype(str))
df_n2t = df_n2t.set_index('Date')
df_n2t = df_n2t.drop(columns=['Year', 'Month'])
df_n2t = df_n2t.sort_index()

df_n2t.head()

In [ ]:
#group by date, borough, then take the mean
df_pt = df_pt.groupby(['Date', 'Borough']).mean()
df_n2t = df_n2t.groupby(['Date', 'Borough']).mean()

df_n2t.head(20)

In [ ]:
#define global variables for min/max dates for df_pt and df_n2t to use in the re-indexing function
global_min_date_pt = df_pt.index.get_level_values('Date').min()
global_max_date_pt = df_pt.index.get_level_values('Date').max()

global_min_date_n2t = df_n2t.index.get_level_values('Date').min()
global_max_date_n2t = df_n2t.index.get_level_values('Date').max()


#define a function to re-index the dataframes by borough and generate new dates

def reindex_borough_data_pt(group):
    borough_name = group.index.get_level_values('Borough').unique()[0]
    full_date_range = pd.date_range(start=global_min_date_pt, end=global_max_date_pt, freq='MS')

    template_df = pd.DataFrame({'Date': full_date_range, 'Borough': borough_name})
    template_df = template_df.set_index(['Date', 'Borough'])

    group_reset = group.reset_index()
    reindexed_group = pd.merge(template_df, group_reset, on=['Date', 'Borough'], how='left')

    reindexed_group = reindexed_group.set_index(['Date', 'Borough']).sort_index()
    return reindexed_group

def reindex_borough_data_n2t(group):
    borough_name = group.index.get_level_values('Borough').unique()[0]
    full_date_range = pd.date_range(start=global_min_date_n2t, end=global_max_date_n2t, freq='MS')

    template_df = pd.DataFrame({'Date': full_date_range, 'Borough': borough_name})
    template_df = template_df.set_index(['Date', 'Borough'])

    group_reset = group.reset_index()
    reindexed_group = pd.merge(template_df, group_reset, on=['Date', 'Borough'], how='left')

    reindexed_group = reindexed_group.set_index(['Date', 'Borough']).sort_index()
    return reindexed_group

In [ ]:
#re-index the dataframes by brough and generate new dates
df_pt_avg_reindexed = df_pt.groupby('Borough', group_keys = False).apply(lambda x: reindex_borough_data_pt(x))
df_n2t_avg_reindexed = df_n2t.groupby('Borough', group_keys = False).apply(lambda x: reindex_borough_data_n2t(x))

df_n2t_avg_reindexed.head()

In [ ]:
print(df_pt_avg_reindexed.describe())
print(df_n2t_avg_reindexed.describe())
df_pt_avg_inter = df_pt_avg_reindexed.interpolate(method='linear')
df_n2t_avg_inter = df_n2t_avg_reindexed.interpolate(method='linear')
print('\n')
print(df_pt_avg_inter.describe())
print(df_n2t_avg_inter.describe())

In [ ]:
print(df_pt_avg_inter[df_pt_avg_inter['Fine_Particles_Log'].isnull()])
print(df_n2t_avg_inter[df_n2t_avg_inter['NO2_Log'].isnull()])

In [ ]:
df_pt_filled = df_pt_avg_inter.groupby('Borough').bfill()
df_n2t_filled = df_n2t_avg_inter.groupby('Borough').bfill()

print(df_pt_filled.head())
print(df_pt_filled.isnull().sum())
print(df_n2t_filled.head())
print(df_n2t_filled.isnull().sum())

In [ ]:
#create lagged features for fine particles and traffic volume
df_pt_filled['Fine_Particles_Log_Lag1'] = df_pt_filled['Fine_Particles_Log'].shift(1)
df_pt_filled['Fine_Particles_Log_Lag12'] = df_pt_filled['Fine_Particles_Log'].shift(12)
df_pt_filled['Traffic_Volume_Log_Lag1'] = df_pt_filled['Traffic_Volume_Log'].shift(1)
df_pt_filled['Traffic_Volume_Log_Lag12'] = df_pt_filled['Traffic_Volume_Log'].shift(12)
df_pt_filled.head(15)

In [ ]:
#create lagged features for NO2 and traffic volume
df_n2t_filled['NO2_Log_Lag1'] = df_n2t_filled['NO2_Log'].shift(1)
df_n2t_filled['NO2_Log_Lag12'] = df_n2t_filled['NO2_Log'].shift(12)
df_n2t_filled['Traffic_Volume_Log_Lag1'] = df_n2t_filled['Traffic_Volume_Log'].shift(1)
df_n2t_filled['Traffic_Volume_Log_Lag12'] = df_n2t_filled['Traffic_Volume_Log'].shift(12)
df_n2t_filled.head()

In [ ]:
#fill NaN values after lagged features are implemented for particles and traffic volume
lagged_pt_cols = ['Fine_Particles_Log_Lag1', 'Fine_Particles_Log_Lag12', 'Traffic_Volume_Log_Lag1', 'Traffic_Volume_Log_Lag12']
for col in lagged_pt_cols:
    df_pt_filled[col] = df_pt_filled.groupby('Borough')[col].bfill()

In [ ]:
#fill NaN values afteer lagged features are implemented for NO2 and traffic volume
lagged_pt_cols = ['NO2_Log_Lag1', 'NO2_Log_Lag12', 'Traffic_Volume_Log_Lag1', 'Traffic_Volume_Log_Lag12']
for col in lagged_pt_cols:
    df_n2t_filled[col] = df_n2t_filled.groupby('Borough')[col].bfill()

In [ ]:
#create time-based features for fine particles and traffic volume
df_pt_filled['Month'] = df_pt_filled.index.get_level_values('Date').month
df_pt_filled['Year'] = df_pt_filled.index.get_level_values('Date').year

#implement cyclical encoding for the month feature
df_pt_filled['Month_Sin'] = np.sin(2 * np.pi * df_pt_filled['Month'] / 12)
df_pt_filled['Month_Cos'] = np.cos(2 * np.pi * df_pt_filled['Month'] / 12)

df_pt_filled.head()

In [ ]:
#create time-based features for no2 particles and traffic volume
df_n2t_filled['Month'] = df_n2t_filled.index.get_level_values('Date').month
df_n2t_filled['Year'] = df_n2t_filled.index.get_level_values('Date').year

#implement cyclical encoding for the month feature
df_n2t_filled['Month_Sin'] = np.sin(2 * np.pi * df_n2t_filled['Month'] / 12)
df_n2t_filled['Month_Cos'] = np.cos(2 * np.pi * df_n2t_filled['Month'] / 12)

df_n2t_filled.head()

In [ ]:
#create a rolling features window for particles and traffic volume
rolling_windows = [3,6,12]

for window in rolling_windows:
    particle_column_name = f'Fine_Particle_RollMean{window}'
    traffic_column_name = f'Traffic_Volume_RollStd{window}'
    df_pt_filled[particle_column_name] = df_pt_filled.groupby('Borough')['Fine_Particles_Log'].rolling(window=window).mean().reset_index(level = 0, drop = True)
    df_pt_filled[traffic_column_name] = df_pt_filled.groupby('Borough')['Traffic_Volume_Log'].rolling(window=window).std().reset_index(level=0, drop=True)
    
df_pt_filled.head()

In [ ]:
#create a rolling features window for NO2 and traffic volume
rolling_windows = [3,6,12]

for window in rolling_windows:
    no2_column_name = f'NO2_RollMean{window}'
    traffic_column_name = f'Traffic_Volume_RollStd{window}'
    df_n2t_filled[no2_column_name] = df_n2t_filled.groupby('Borough')['NO2_Log'].rolling(window=window).mean().reset_index(level = 0, drop = True)
    df_n2t_filled[traffic_column_name] = df_n2t_filled.groupby('Borough')['Traffic_Volume_Log'].rolling(window=window).std().reset_index(level=0, drop=True)
    
df_n2t_filled.head()


In [ ]:
#fill in NaN values after rolling features window for particles and traffic volume
rolling_pt_cols = ['Fine_Particle_RollMean3', 'Fine_Particle_RollMean6', 'Fine_Particle_RollMean12', 'Traffic_Volume_RollStd3', 'Traffic_Volume_RollStd6', 'Traffic_Volume_RollStd12']

for col in rolling_pt_cols:
    df_pt_filled[col] = df_pt_filled.groupby('Borough')[col].bfill()

In [ ]:
df_pt_filled.head()

In [ ]:
#fill in NaN values after rolling features window for NO2 and traffic volume
rolling_n2t_cols = ['NO2_RollMean3', 'NO2_RollMean6', 'NO2_RollMean12', 'Traffic_Volume_RollStd3', 'Traffic_Volume_RollStd6', 'Traffic_Volume_RollStd12']

for col in rolling_n2t_cols:
    df_n2t_filled[col] = df_n2t_filled.groupby('Borough')[col].bfill()

In [ ]:
df_n2t_filled.head()

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
#check stationarity with ADF test for fine particles
result_pt = adfuller(df_pt_filled['Fine_Particles_Log'])
print('ADF Statistic (Fine_Particles_Log): %f' % result_pt[0])
print('p-value (Fine_Particles_Log): %f' % result_pt[1])
print('Critical Values:')
for key, value in result_pt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
#check stationarity with ADF test foe no2
result_n2t = adfuller(df_n2t_filled['NO2_Log'])
print('ADF Statistic (NO2_Log): %f' % result_n2t[0])
print('p-value (NO2_Log): %f' % result_n2t[1])
print('Critical Values:')
for key, value in result_n2t[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
#make fine particles log data stationary
df_pt_filled['Fine_Particles_Log_Diff'] = df_pt_filled['Fine_Particles_Log'].diff()

In [ ]:
#make no2 log data stationary
df_n2t_filled['NO2_Log_Diff'] = df_n2t_filled['NO2_Log'].diff()

In [ ]:
#check the results for fine particles using the differed log
result_pt_diff = adfuller(df_pt_filled['Fine_Particles_Log_Diff'].dropna())
print('ADF Statistic (Fine_Particles_Log_Diff): %f' % result_pt_diff[0])
print('p-value (Fine_Particles_Log_Diff): %f' % result_pt_diff[1])
print('Critical Values:')
for key, value in result_pt_diff[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
#check the results for no2 using the differed log
result_n2t_diff = adfuller(df_n2t_filled['NO2_Log_Diff'].dropna())
print('ADF Statistic (NO2_Log_Diff): %f' % result_n2t_diff[0])
print('p-value (NO2_Log_Diff): %f' % result_n2t_diff[1])
print('Critical Values:')
for key, value in result_n2t_diff[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
# Check stationarity for Traffic_Volume_Log in df_pt_filled
result_traffic_pt = adfuller(df_pt_filled['Traffic_Volume_Log'])
print('ADF Statistic (Traffic_Volume_Log in df_pt_filled): %f' % result_traffic_pt[0])
print('p-value (Traffic_Volume_Log in df_pt_filled): %f' % result_traffic_pt[1])
print('Critical Values:')
for key, value in result_traffic_pt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
# Check stationarity for Traffic_Volume_Log in df_n2t_filled
result_traffic_n2t = adfuller(df_n2t_filled['Traffic_Volume_Log'])
print('ADF Statistic (Traffic_Volume_Log in df_n2t_filled): %f' % result_traffic_n2t[0])
print('p-value (Traffic_Volume_Log in df_n2t_filled): %f' % result_traffic_n2t[1])
print('Critical Values:')
for key, value in result_traffic_n2t[4].items():
    print('\t%s: %.3f' % (key, value))


In [ ]:
# Define the target variable
target_pt = 'Fine_Particles_Log_Diff'

# Define the features to check correlation against
# Exclude the original Fine_Particles_Log as we are predicting its differenced version
features_pt = [
    'Traffic_Volume_Log',
    'Fine_Particles_Log_Lag1',
    'Fine_Particles_Log_Lag12',
    'Traffic_Volume_Log_Lag1',
    'Traffic_Volume_Log_Lag12',
    'Month_Sin',
    'Month_Cos',
    'Fine_Particle_RollMean3',
    'Traffic_Volume_RollStd3',
    'Fine_Particle_RollMean6',
    'Traffic_Volume_RollStd6',
    'Fine_Particle_RollMean12',
    'Traffic_Volume_RollStd12'
]

# Create a temporary DataFrame with only the relevant columns and drop NaNs
correlation_df_pt = df_pt_filled[[target_pt] + features_pt].dropna()

# Calculate correlations of the target with all features
correlations_pt = correlation_df_pt.corr()[target_pt].sort_values(ascending=False)

# Print the correlations (excluding the target correlating with itself)
print(f"Correlations with {target_pt}:\n{correlations_pt.drop(target_pt)}")

In [ ]:
# Define the target variable
target_n2t = 'NO2_Log_Diff'

# Define the features to check correlation against
# Exclude the original Fine_Particles_Log as we are predicting its differenced version
features_n2t = [
    'Traffic_Volume_Log',
    'NO2_Log_Lag1',
    'NO2_Log_Lag12',
    'Traffic_Volume_Log_Lag1',
    'Traffic_Volume_Log_Lag12',
    'Month_Sin',
    'Month_Cos',
    'NO2_RollMean3',
    'Traffic_Volume_RollStd3',
    'NO2_RollMean6',
    'Traffic_Volume_RollStd6',
    'NO2_RollMean12',
    'Traffic_Volume_RollStd12'
]

# Create a temporary DataFrame with only the relevant columns and drop NaNs
correlation_df_n2t = df_n2t_filled[[target_n2t] + features_n2t].dropna()

# Calculate correlations of the target with all features
correlations_n2t = correlation_df_n2t.corr()[target_n2t].sort_values(ascending=False)

# Print the correlations (excluding the target correlating with itself)
print(f"Correlations with {target_n2t}:\n{correlations_n2t.drop(target_n2t)}")

In [ ]:
target_pt = 'Fine_Particles_Log_Diff'
#defined features after correlation analysis
fine_particles_features = [
    'Fine_Particles_Log_Lag1',
    'Fine_Particles_Log_Lag12',
    'Month_Sin',
    'Month_Cos',
    'Traffic_Volume_Log_Lag1',
    'Traffic_Volume_Log_Lag12'
]

# create the modeling DataFrame for fine particles
df_pt_model = df_pt_filled[[target_pt] + fine_particles_features].dropna()

print(f"Shape of df_pt_model: {df_pt_model.shape}")
print(f"First 5 rows of df_pt_model:\n{df_pt_model.head()}")

In [ ]:
# Define the target and features for NO2
target_n2t = 'NO2_Log_Diff'
no2_features = [
    'Month_Cos',
    'Month_Sin',
    'NO2_Log_Lag1',
    'NO2_Log_Lag12',
    'Traffic_Volume_Log',
    'Traffic_Volume_Log_Lag1',
    'Traffic_Volume_Log_Lag12'
]

# Create the modeling DataFrame for NO2
df_n2t_model = df_n2t_filled[[target_n2t] + no2_features].dropna()

print(f"Shape of df_n2t_model: {df_n2t_model.shape}")
print(f"First 5 rows of df_n2t_model:\n{df_n2t_model.head()}")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

series_to_plot = df_pt_model['Fine_Particles_Log_Diff'].dropna()
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Plot ACF
plot_acf(series_to_plot, lags=30, ax=axes[0]) # lags can be adjusted
axes[0].set_title('Autocorrelation Function (ACF)')

# Plot PACF
plot_pacf(series_to_plot, lags=30, ax=axes[1]) # lags can be adjusted
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

In [ ]:
series_to_plot = df_n2t_model['NO2_Log_Diff'].dropna()
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Plot ACF
plot_acf(series_to_plot, lags=30, ax=axes[0]) # lags can be adjusted
axes[0].set_title('Autocorrelation Function (ACF)')

# Plot PACF
plot_pacf(series_to_plot, lags=30, ax=axes[1]) # lags can be adjusted
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

In [ ]:
# Define the split ratio
split_ratio = 0.8

# Get unique dates from the index
unique_dates_pt = df_pt_model.index.get_level_values('Date').unique()

# Calculate the split date index
split_date_index_pt = int(len(unique_dates_pt) * split_ratio)

# Get the actual split date
split_date_pt = unique_dates_pt[split_date_index_pt]

print(f"Fine Particles - Split Date: {split_date_pt}")

# Split the data for fine particles based on the split date
X_train_pt = df_pt_model[df_pt_model.index.get_level_values('Date') < split_date_pt][fine_particles_features]
X_test_pt = df_pt_model[df_pt_model.index.get_level_values('Date') >= split_date_pt][fine_particles_features]
y_train_pt = df_pt_model[df_pt_model.index.get_level_values('Date') < split_date_pt][target_pt]
y_test_pt = df_pt_model[df_pt_model.index.get_level_values('Date') >= split_date_pt][target_pt]

print(f"Fine Particles - Training features shape: {X_train_pt.shape}")
print(f"Fine Particles - Testing features shape: {X_test_pt.shape}")
print(f"Fine Particles - Training target shape: {y_train_pt.shape}")
print(f"Fine Particles - Testing target shape: {y_test_pt.shape}")

In [ ]:
# Define the split ratio (already defined as 0.8)
# split_ratio = 0.8

# Get unique dates from the index
unique_dates_n2t = df_n2t_model.index.get_level_values('Date').unique()

# Calculate the split date index
split_date_index_n2t = int(len(unique_dates_n2t) * split_ratio)

# Get the actual split date
split_date_n2t = unique_dates_n2t[split_date_index_n2t]

print(f"NO2 - Split Date: {split_date_n2t}")

# Split the data for NO2 based on the split date
X_train_n2t = df_n2t_model[df_n2t_model.index.get_level_values('Date') < split_date_n2t][no2_features]
X_test_n2t = df_n2t_model[df_n2t_model.index.get_level_values('Date') >= split_date_n2t][no2_features]
y_train_n2t = df_n2t_model[df_n2t_model.index.get_level_values('Date') < split_date_n2t][target_n2t]
y_test_n2t = df_n2t_model[df_n2t_model.index.get_level_values('Date') >= split_date_n2t][target_n2t]

print(f"NO2 - Training features shape: {X_train_n2t.shape}")
print(f"NO2 - Testing features shape: {X_test_n2t.shape}")
print(f"NO2 - Training target shape: {y_train_n2t.shape}")
print(f"NO2 - Testing target shape: {y_test_n2t.shape}")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
#create time-series model for fine particles
order_pt = (1,1,1)
seasonal_order_pt = (1,0, 1, 12)

unique_boroughs_pt = df_pt_model.index.get_level_values('Borough').unique()
all_borough_models_pt = {} # To store models for each borough
all_borough_results_pt = {} # To store results for each borough

exog_features_pt = fine_particles_features

for borough in unique_boroughs_pt:
    print(f"Fitting SARIMAX for Fine_Particles_Log_Diff in {borough}...")
    borough_data = df_pt_model.loc[(slice(None), borough), :]
    
    endog_series = borough_data['Fine_Particles_Log_Diff'].reset_index(level='Borough', drop=True)
    endog_series.index = pd.to_datetime(endog_series.index)
    endog_series = endog_series.asfreq('MS')

    exog_df = borough_data[exog_features_pt].reset_index(level='Borough', drop=True)
    exog_df.index = pd.to_datetime(exog_df.index)
    exog_df = exog_df.asfreq('MS') 

    model_pt_borough = SARIMAX(
        endog=endog_series,
        exog=exog_df,
        order=order_pt,
        seasonal_order=seasonal_order_pt
    )

    results_pt_borough = model_pt_borough.fit(maxiter=1000, disp=False, low_memory=True)
    print(f"Finished fitting for {borough}.\n")

    all_borough_models_pt[borough] = model_pt_borough
    all_borough_results_pt[borough] = results_pt_borough



In [ ]:
#create time-series model for no2
order_n2t = (1,1,1)
seasonal_order_n2t = (1,0,1,12)

unique_boroughs_n2t = df_n2t_model.index.get_level_values('Borough').unique()
all_borough_models_n2t = {} # To store models for each borough
all_borough_results_n2t = {} # To store results for each borough

exog_features_n2t = no2_features

for borough in unique_boroughs_n2t:
    print(f"Fitting SARIMAX for NO2_Log_Diff in {borough}...")
    borough_data = df_n2t_model.loc[(slice(None), borough), :]
    
    endog_series = borough_data['NO2_Log_Diff'].reset_index(level='Borough', drop=True)
    endog_series.index = pd.to_datetime(endog_series.index)
    endog_series = endog_series.asfreq('MS')

    exog_df = borough_data[exog_features_n2t].reset_index(level='Borough', drop=True)
    exog_df.index = pd.to_datetime(exog_df.index)
    exog_df = exog_df.asfreq('MS') 

    model_n2t_borough = SARIMAX(
        endog=endog_series,
        exog=exog_df,
        order=order_n2t,
        seasonal_order=seasonal_order_n2t
    )

    results_n2t_borough = model_n2t_borough.fit(maxiter=1000, disp=False, low_memory=True)
    print(f"Finished fitting for {borough}.\n")

    all_borough_models_n2t[borough] = model_n2t_borough
    all_borough_results_n2t[borough] = results_n2t_borough


In [ ]:
all_borough_predictions_pt = {} # To store predictions for each borough

for borough in unique_boroughs_pt:
    print(f"Generating predictions for Fine_Particles_Log_Diff in {borough}...")

    # Retrieve the fitted model results for this borough
    borough_results = all_borough_results_pt[borough]

    # Get the data for this borough from your original df_pt_model
    borough_data = df_pt_model.loc[(slice(None), borough), :]

    # Separate the exogenous features for the test period for this borough
    # Using the dynamically calculated split_date_pt
    borough_exog_test = borough_data[fine_particles_features].loc[split_date_pt:]
    borough_exog_test.index = pd.to_datetime(borough_exog_test.index.get_level_values('Date'))
    borough_exog_test = borough_exog_test.asfreq('MS')

    # Generate predictions
    # The 'start' and 'end' dates should correspond to your test set's date range
    borough_predictions = borough_results.predict(
        start=borough_exog_test.index[0],
        end=borough_exog_test.index[-1],
        exog=borough_exog_test
    )

    all_borough_predictions_pt[borough] = borough_predictions
    print(f"Finished predictions for {borough}.\n")


In [ ]:
all_borough_predictions_n2t = {} # To store predictions for each borough

for borough in unique_boroughs_n2t:
    print(f"Generating predictions for NO2_Log_Diff in {borough}...")

    # Retrieve the fitted model results for this borough
    borough_results = all_borough_results_n2t[borough]

    # Get the data for this borough from the original df_n2t_model
    borough_data = df_n2t_model.loc[(slice(None), borough), :]

    # Separate the exogenous features for the test period for this borough
    # Using the dynamically calculated split_date_n2t
    borough_exog_test = borough_data[no2_features].loc[split_date_n2t:]
    borough_exog_test.index = pd.to_datetime(borough_exog_test.index.get_level_values('Date'))
    borough_exog_test = borough_exog_test.asfreq('MS')

    # Generate predictions
    # The 'start' and 'end' dates should correspond to your test set's date range
    borough_predictions = borough_results.predict(
        start=borough_exog_test.index[0],
        end=borough_exog_test.index[-1],
        exog=borough_exog_test
    )

    all_borough_predictions_n2t[borough] = borough_predictions
    print(f"Finished predictions for {borough}.\n")


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
all_borough_metrics_pt = {} # To store all metrics for each borough

print("Calculating MAE, MSE, and RMSE for each borough...")

for borough in unique_boroughs_pt:
    # Get the actual test values for the current borough
    # This should already return a Series indexed by 'Date'
    actual_values = y_test_pt.loc[(slice(None), borough)]
    actual_values.index = pd.to_datetime(actual_values.index) # Ensure index is DatetimeIndex

    # Get the predicted values for the current borough
    predicted_values = all_borough_predictions_pt[borough]

    # Align actual and predicted values by their index (Date)
    aligned_actual, aligned_predicted = actual_values.align(predicted_values, join='inner')

    if not aligned_actual.empty:
        # Calculate metrics
        mae = mean_absolute_error(aligned_actual, aligned_predicted)
        mse = mean_squared_error(aligned_actual, aligned_predicted)
        rmse = np.sqrt(mse)

        all_borough_metrics_pt[borough] = {
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse
        }
        print(f"\nMetrics for {borough}:")
        print(f"  MAE: {mae:.4f}")
        print(f"  MSE: {mse:.4f}")
        print(f"  RMSE: {rmse:.4f}")
    else:
        print(f"\nNo overlapping dates for {borough} between actual and predicted values. Skipping metric calculation.")

print("\nFinished calculating all metrics for all boroughs.")

In [ ]:
all_borough_metrics_n2t = {} # To store all metrics for each borough

print("Calculating MAE, MSE, and RMSE for each borough...")

for borough in unique_boroughs_n2t:
    actual_values = y_test_n2t.loc[(slice(None), borough)]
    actual_values.index = pd.to_datetime(actual_values.index) # Ensure index is DatetimeIndex

    # Get the predicted values for the current borough
    predicted_values = all_borough_predictions_n2t[borough]

    # Align actual and predicted values by their index (Date)
    aligned_actual, aligned_predicted = actual_values.align(predicted_values, join='inner')

    if not aligned_actual.empty:
        # Calculate metrics
        mae = mean_absolute_error(aligned_actual, aligned_predicted)
        mse = mean_squared_error(aligned_actual, aligned_predicted)
        rmse = np.sqrt(mse)

        all_borough_metrics_n2t[borough] = {
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse
        }
        print(f"\nMetrics for {borough}:")
        print(f"  MAE: {mae:.4f}")
        print(f"  MSE: {mse:.4f}")
        print(f"  RMSE: {rmse:.4f}")
    else:
        print(f"\nNo overlapping dates for {borough} between actual and predicted values. Skipping metric calculation.")

print("\nFinished calculating all metrics for all boroughs.")

In [ ]:
import math

In [ ]:
print("Generating plots for all boroughs...")

num_boroughs_pt = len(unique_boroughs_pt)
# Determine the grid size for subplots (e.g., 2x3, 3x2, etc.)
# A simple way is to calculate rows and columns based on the number of boroughs
rows = math.ceil(num_boroughs_pt/ 2) # Example: 2 columns per row
cols = 2

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), squeeze=False)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, borough in enumerate(unique_boroughs_pt):
    ax = axes[i]

    # Get the actual test values for the current borough
    actual_values_plot_pt = y_test_pt.loc[(slice(None), borough)]
    actual_values_plot_pt.index = pd.to_datetime(actual_values_plot_pt.index)

    # Get the predicted values for the current borough
    predicted_values_plot_pt = all_borough_predictions_pt[borough]

    # Align actual and predicted values by their index (Date)
    aligned_actual_plot_pt, aligned_predicted_plot_pt = actual_values_plot_pt.align(predicted_values_plot_pt, join='inner')

    if not aligned_actual_plot_pt.empty:
        ax.plot(aligned_actual_plot_pt.index, aligned_actual_plot_pt.values, label='Actual Values', color='blue')
        ax.plot(aligned_predicted_plot_pt.index, aligned_predicted_plot_pt.values, label='Predicted Values', color='red', linestyle='--')
        ax.set_title(f'{borough}')
        ax.set_xlabel('Date')
        ax.set_ylabel('Fine_Particles_Log_Diff')
        ax.legend()
        ax.grid(True)
    else:
        ax.set_title(f'{borough} (No overlapping data)')
        ax.text(0.5, 0.5, 'No data to plot', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print("Plotting complete for all boroughs.")

In [ ]:
print("Generating plots for all boroughs...")

num_boroughs_n2t = len(unique_boroughs_n2t)
# A simple way is to calculate rows and columns based on the number of boroughs
rows = math.ceil(num_boroughs_n2t/ 2) # Example: 2 columns per row
cols = 2

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), squeeze=False)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, borough in enumerate(unique_boroughs_n2t):
    ax = axes[i]

    # Get the actual test values for the current borough
    actual_values_plot_n2t = y_test_n2t.loc[(slice(None), borough)]
    actual_values_plot_n2t.index = pd.to_datetime(actual_values_plot_n2t.index)

    # Get the predicted values for the current borough
    predicted_values_plot_n2t = all_borough_predictions_n2t[borough]

    # Align actual and predicted values by their index (Date)
    aligned_actual_plot_n2t, aligned_predicted_plot_n2t = actual_values_plot_n2t.align(predicted_values_plot_n2t, join='inner')

    if not aligned_actual_plot_n2t.empty:
        ax.plot(aligned_actual_plot_n2t.index, aligned_actual_plot_n2t.values, label='Actual Values', color='blue')
        ax.plot(aligned_predicted_plot_n2t.index, aligned_predicted_plot_n2t.values, label='Predicted Values', color='red', linestyle='--')
        ax.set_title(f'{borough}')
        ax.set_xlabel('Date')
        ax.set_ylabel('NO2_Log_Diff')
        ax.legend()
        ax.grid(True)
    else:
        ax.set_title(f'{borough} (No overlapping data)')
        ax.text(0.5, 0.5, 'No data to plot', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print("Plotting complete for all boroughs.")

In [ ]:
print("Generating overall NYC plot...")

# 1. Calculate the average actual values across all boroughs for each date
nyc_actual_avg_pt = y_test_pt.groupby(level='Date').mean()
nyc_actual_avg_pt.index = pd.to_datetime(nyc_actual_avg_pt.index)

# 2. Combine all predicted borough series into a DataFrame and calculate the average across boroughs for each date
# First, ensure all predicted series are aligned by date before averaging
predicted_df_pt = pd.DataFrame(all_borough_predictions_pt)
nyc_predicted_avg_pt = predicted_df_pt.mean(axis=1) # Mean across columns (boroughs) for each date
nyc_predicted_avg_pt.index = pd.to_datetime(nyc_predicted_avg_pt.index)

# 3. Align the overall actual and predicted values by their index (Date)
aligned_nyc_actual_pt, aligned_nyc_predicted_pt = nyc_actual_avg_pt.align(nyc_predicted_avg_pt, join='inner')

if not aligned_nyc_actual_pt.empty:
    plt.figure(figsize=(14, 7))
    plt.plot(aligned_nyc_actual_pt.index, aligned_nyc_actual_pt.values, label='Overall NYC Actual Average', color='darkblue')
    plt.plot(aligned_nyc_predicted_pt.index, aligned_nyc_predicted_pt.values, label='Overall NYC Predicted Average', color='darkred', linestyle='--')
    plt.title('Overall NYC: Actual vs. Predicted Average Fine_Particles_Log_Diff')
    plt.xlabel('Date')
    plt.ylabel('Average Fine_Particles_Log_Diff')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No overlapping dates to plot for overall NYC.")

print("Overall NYC plotting complete.")

In [ ]:
print("Generating overall NYC plot...")

# 1. Calculate the average actual values across all boroughs for each date
nyc_actual_avg_n2t = y_test_n2t.groupby(level='Date').mean()
nyc_actual_avg_n2t.index = pd.to_datetime(nyc_actual_avg_n2t.index)

# 2. Combine all predicted borough series into a DataFrame and calculate the average across boroughs for each date
# First, ensure all predicted series are aligned by date before averaging
predicted_df_n2t = pd.DataFrame(all_borough_predictions_n2t)
nyc_predicted_avg_n2t = predicted_df_n2t.mean(axis=1) # Mean across columns (boroughs) for each date
nyc_predicted_avg_n2t.index = pd.to_datetime(nyc_predicted_avg_n2t.index)

# 3. Align the overall actual and predicted values by their index (Date)
aligned_nyc_actual_n2t, aligned_nyc_predicted_n2t = nyc_actual_avg_n2t.align(nyc_predicted_avg_n2t, join='inner')

if not aligned_nyc_actual_n2t.empty:
    plt.figure(figsize=(14, 7))
    plt.plot(aligned_nyc_actual_n2t.index, aligned_nyc_actual_n2t.values, label='Overall NYC Actual Average', color='darkblue')
    plt.plot(aligned_nyc_predicted_n2t.index, aligned_nyc_predicted_n2t.values, label='Overall NYC Predicted Average', color='darkred', linestyle='--')
    plt.title('Overall NYC: Actual vs. Predicted Average NO2_Log_Diff')
    plt.xlabel('Date')
    plt.ylabel('Average NO2_Log_Diff')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No overlapping dates to plot for overall NYC.")

print("Overall NYC plotting complete.")

In [ ]:
def plot_borough_residuals(borough_name, y_test_data, predictions_data):
    """
    Calculates and plots the residuals (Actual - Predicted) for a given borough.

    Args:
        borough_name (str): The name of the borough to analyze.
        y_test_data (pd.Series): The actual test values (e.g., y_test_pt).
        predictions_data (dict): A dictionary of predicted values for all boroughs
                                 (e.g., all_borough_predictions_pt).
    """
    print(f"Calculating and plotting residuals for {borough_name}...")

    # 1. Get the actual test values for the specified borough
    actual_borough = y_test_data.loc[(slice(None), borough_name)]
    actual_borough.index = pd.to_datetime(actual_borough.index)

    # 2. Get the predicted values for the specified borough
    predicted_borough = predictions_data[borough_name]
    predicted_borough.index = pd.to_datetime(predicted_borough.index)

    # 3. Align actual and predicted values by their index (Date)
    aligned_actual, aligned_predicted = actual_borough.align(predicted_borough, join='inner')

    if not aligned_actual.empty:
        # 4. Calculate the residuals: Actual - Predicted
        residuals = aligned_actual - aligned_predicted

        # 5. Plot the residuals
        plt.figure(figsize=(12, 6))
        plt.plot(residuals.index, residuals.values, label='Residuals (Actual - Predicted)', color='purple', alpha=0.7)
        plt.axhline(0, color='black', linestyle='--', linewidth=0.8, label='Zero Error Line')
        plt.title(f'Residuals for {borough_name}: Actual vs. Predicted Fine_Particles_Log_Diff')
        plt.xlabel('Date')
        plt.ylabel('Residual Value')
        plt.legend()
        plt.grid(True)
        plt.show()

        # 6. Print summary statistics for the residuals
        print(f"\nResiduals Summary for {borough_name}:\n{residuals.describe()}")
    else:
        print(f"No overlapping data to calculate residuals for {borough_name}.")

    print(f"Residuals plotting complete for {borough_name}.")

# Example of how to use the function (you can change 'Staten Island' to any borough)
# plot_borough_residuals('Staten Island', y_test_pt, all_borough_predictions_pt)

In [ ]:
all_borough_names_pt = y_test_pt.index.get_level_values(1).unique().tolist()

print("Generating residual plots for all boroughs Fine Particles...")

# Now, loop through each borough and call the function
for borough in all_borough_names_pt:
    plot_borough_residuals(borough, y_test_pt, all_borough_predictions_pt)

print("All borough residual plots generated!")

In [ ]:
all_borough_names_n2t = y_test_n2t.index.get_level_values(1).unique().tolist()

print("Generating residual plots for all boroughs NO2...")

# Now, loop through each borough and call the function
for borough in all_borough_names_n2t:
    plot_borough_residuals(borough, y_test_n2t, all_borough_predictions_n2t)

print("All borough residual plots generated!")